In [1]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, LSTM, TimeDistributed, Dropout
from tensorflow.keras.optimizers import Adam

In [2]:
# Setting image and sequence variable

base_path = 'Drowsy_dataset' 
img_height, img_width = 64, 64
sequence_length = 5  # Num of frames per sequence


In [3]:
# قراءة الصور من مجلد train أو test وتجهيز sequences

def load_sequences_from_folder(folder_path, sequence_length=5):
    X_sequences = []
    y_sequences = []

    classes = sorted(os.listdir(folder_path))  # ['DROWSY', 'NATURAL']
    print("Classes found:", classes)

    for class_idx, class_name in enumerate(classes):
        class_path = os.path.join(folder_path, class_name)
        if not os.path.isdir(class_path):
            continue
        images = sorted(os.listdir(class_path))
        temp_seq = []

        for img_name in images:
            img_path = os.path.join(class_path, img_name)
            img = load_img(img_path, target_size=(img_height, img_width))
            img_array = img_to_array(img) / 255.0  # Normalize
            temp_seq.append(img_array)

            # إذا وصلنا طول الـ sequence نخزنه
            if len(temp_seq) == sequence_length:
                X_sequences.append(temp_seq)
                y_sequences.append(class_idx)
                temp_seq = []

    X_sequences = np.array(X_sequences)
    y_sequences = to_categorical(np.array(y_sequences), num_classes=len(classes))
    return X_sequences, y_sequences


In [4]:
# Read training and test data
 
train_path = os.path.join(base_path, 'train')
X_train, y_train = load_sequences_from_folder(train_path, sequence_length)

test_path = os.path.join(base_path, 'test')
X_test, y_test = load_sequences_from_folder(test_path, sequence_length)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Classes found: ['DROWSY', 'NATURAL']
Classes found: ['DROWSY', 'NATURAL']
Training samples: 1171
Testing samples: 296


In [5]:
# Building a CNN + LSTM model
 
model = Sequential()

# CNN for each frame
model.add(TimeDistributed(Conv2D(32, (3,3), activation='relu'), input_shape=(sequence_length, img_height, img_width, 3)))
model.add(TimeDistributed(MaxPooling2D((2,2))))
model.add(TimeDistributed(Conv2D(64, (3,3), activation='relu')))
model.add(TimeDistributed(MaxPooling2D((2,2))))
model.add(TimeDistributed(Conv2D(128, (3,3), activation='relu')))
model.add(TimeDistributed(MaxPooling2D((2,2))))
model.add(TimeDistributed(Flatten()))

# LSTM 
model.add(LSTM(64))
model.add(Dropout(0.5))
model.add(Dense(2, activation='softmax'))  # DROWSY vs NATURAL


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [6]:
# Compile Model
 
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ time_distributed (TimeDistributed)   │ (None, 5, 62, 62, 32)       │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_1 (TimeDistributed) │ (None, 5, 31, 31, 32)       │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_2 (TimeDistributed) │ (None, 5, 29, 29, 64)       │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_3 (TimeDistributed) │ (None, 5, 14, 14, 64)       │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_4 (TimeDistributed) │ (None, 5, 12, 12, 128)      │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_5 (TimeDistributed) │ (None, 5, 6, 6, 128)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_6 (TimeDistributed) │ (None, 5, 4608)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 64)                  │       1,196,288 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 2)                   │             130 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,289,666 (4.92 MB)

 Trainable params: 1,289,666 (4.92 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# training
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=16
)

# testing
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {acc:.4f}")

Epoch 1/20
74/74 ━━━━━━━━━━━━━━━━━━━━ 21s 248ms/step - accuracy: 0.5949 - loss: 0.6073 - val_accuracy: 0.8784 - val_loss: 0.2281
Epoch 2/20
74/74 ━━━━━━━━━━━━━━━━━━━━ 19s 254ms/step - accuracy: 0.9167 - loss: 0.1938 - val_accuracy: 0.9831 - val_loss: 0.0346
Epoch 3/20
74/74 ━━━━━━━━━━━━━━━━━━━━ 23s 304ms/step - accuracy: 0.9497 - loss: 0.1083 - val_accuracy: 0.9358 - val_loss: 0.2157
Epoch 4/20
74/74 ━━━━━━━━━━━━━━━━━━━━ 21s 276ms/step - accuracy: 0.9872 - loss: 0.0487 - val_accuracy: 0.9358 - val_loss: 0.2149
Epoch 5/20
74/74 ━━━━━━━━━━━━━━━━━━━━ 20s 272ms/step - accuracy: 0.9931 - loss: 0.0172 - val_accuracy: 0.9628 - val_loss: 0.1337
Epoch 6/20
74/74 ━━━━━━━━━━━━━━━━━━━━ 20s 270ms/step - accuracy: 0.9965 - loss: 0.0112 - val_accuracy: 0.9392 - val_loss: 0.2426
Epoch 7/20
74/74 ━━━━━━━━━━━━━━━━━━━━ 20s 273ms/step - accuracy: 0.9983 - loss: 0.0154 - val_accuracy: 0.9595 - val_loss: 0.1130
Epoch 8/20
74/74 ━━━━━━━━━━━━━━━━━━━━ 20s 265ms/step - accuracy: 0.9971 - loss: 0.0108 - val_accu